# 03 — Bearing Applications

Bearing is only useful when it connects to something real.

This notebook applies `compute_bearing` to practical problems: reading direction from a map click, filtering targets by heading sector, drawing direction arrows, and combining bearing with distance to build a simple targeting picture.

The geometry is the same as before. The context is what makes it matter.

In [1]:
import math
import json
from pathlib import Path

def compute_bearing(p1, p2):
    """Bearing from p1 to p2. Inputs: [lon, lat]. Output: degrees [0, 360)."""
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    """Great-circle distance in km. Inputs: [lon, lat]."""
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

with open(Path("data/bearing_examples.json")) as f:
    examples = json.load(f)

print("Ready.")

Ready.


## 1. Direction from a Map Click

The most natural bearing question on an interactive map is: *I clicked two points — what direction is the second from the first?*

The `on_interaction` handler from the distance notebook applies directly here. This version tracks two successive clicks and computes both the bearing and the distance between them.

In [2]:
from ipyleaflet import Map, GeoJSON, Marker
import ipywidgets as widgets

click_state = {"points": [], "layers": []}
output = widgets.Output()

m = Map(center=(34.5, -97.5), zoom=6)

def bearing_to_label(bearing):
    """Convert a bearing in degrees to the nearest compass label."""
    labels = ["N","NNE","NE","ENE","E","ESE","SE","SSE",
              "S","SSW","SW","WSW","W","WNW","NW","NNW"]
    idx = round(bearing / 22.5) % 16
    return labels[idx]

def handle_click(**kwargs):
    if kwargs.get("type") != "click":
        return

    lat, lon = kwargs["coordinates"]
    pt = [lon, lat]
    click_state["points"].append(pt)

    # Remove old layers
    for layer in click_state["layers"]:
        m.remove(layer)
    click_state["layers"].clear()

    # Add marker for current click
    marker = Marker(location=(lat, lon))
    m.add(marker)
    click_state["layers"].append(marker)

    pts = click_state["points"]

    if len(pts) >= 2:
        p1, p2 = pts[-2], pts[-1]
        brg  = compute_bearing(p1, p2)
        dist = haversine_km(p1, p2)
        lbl  = bearing_to_label(brg)

        # Draw line between the two points
        line = GeoJSON(data={
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [p1, p2]},
            "properties": {}
        }, style={"color": "#e63946", "weight": 2})
        m.add(line)
        click_state["layers"].append(line)

        with output:
            output.clear_output()
            print(f"From:    [{p1[0]:.4f}, {p1[1]:.4f}]")
            print(f"To:      [{p2[0]:.4f}, {p2[1]:.4f}]")
            print(f"Bearing: {brg:.1f}°  ({lbl})")
            print(f"Distance: {dist:.1f} km")

m.on_interaction(handle_click)
widgets.VBox([m, output])

Click anywhere on the map twice. The bearing and distance between the two clicks print below the map, along with the nearest 16-point compass label.

## 2. Bearing Sector Filtering

A bearing sector is a angular window: "targets between 30° and 90°" means anything in the northeast quadrant from the origin. Filtering by sector is how you answer *which targets are roughly in front of me?*

The function below returns all features from a FeatureCollection whose bearing from an origin falls within a given sector. It handles the wraparound case (e.g. a sector from 340° to 20° that spans due North).

In [3]:
def in_sector(origin, features, start_bearing, end_bearing):
    """
    Return features whose bearing from origin falls within [start_bearing, end_bearing].
    Handles wraparound (e.g. 350° → 20°).

    Parameters
    ----------
    origin        : [lon, lat]
    features      : list of GeoJSON Point features
    start_bearing : float, degrees
    end_bearing   : float, degrees

    Returns
    -------
    list of (feature, bearing) tuples
    """
    results = []
    for feat in features:
        coords = feat["geometry"]["coordinates"]
        brg = compute_bearing(origin, coords)

        if start_bearing <= end_bearing:
            match = start_bearing <= brg <= end_bearing
        else:
            # Sector wraps through North (e.g. 340° → 20°)
            match = brg >= start_bearing or brg <= end_bearing

        if match:
            results.append((feat, brg))

    return sorted(results, key=lambda x: x[1])


# Airfield targets around Wichita Falls
launch = [-98.49, 33.91]  # Wichita Falls

targets_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.37, 35.39]}, "properties": {"name": "Tinker AFB"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-94.37, 35.34]}, "properties": {"name": "Fort Smith Regional"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-101.70, 33.66]}, "properties": {"name": "Lubbock Preston Smith"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.04, 32.85]}, "properties": {"name": "NAS Fort Worth JRB"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.47, 29.53]}, "properties": {"name": "Kelly Field Annex"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-99.27, 34.66]}, "properties": {"name": "Altus AFB"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.52, 35.47]}, "properties": {"name": "Oklahoma City"}},
    ]
}

features = targets_fc["features"]

# Print bearing to each target for reference
print("All targets from Wichita Falls:")
print(f"  {'Name':<26} {'Bearing':>9}  {'Label':>5}  {'Distance':>10}")
print("  " + "-" * 58)
for feat in features:
    coords = feat["geometry"]["coordinates"]
    brg  = compute_bearing(launch, coords)
    dist = haversine_km(launch, coords)
    lbl  = bearing_to_label(brg)
    print(f"  {feat['properties']['name']:<26} {brg:>8.1f}°  {lbl:>5}  {dist:>8.0f} km")

All targets from Wichita Falls:
  Name                         Bearing  Label    Distance
  ----------------------------------------------------------
  Tinker AFB                     31.6°    NNE       194 km
  Fort Smith Regional            66.0°    ENE       409 km
  Lubbock Preston Smith         265.5°      W       298 km
  NAS Fort Worth JRB            130.8°     SE       179 km
  Kelly Field Annex             179.8°      S       487 km
  Altus AFB                     319.5°     NW       110 km
  Oklahoma City                  26.8°    NNE       195 km


In [4]:
# Filter to the northeast sector (0° – 90°) and the south sector (135° – 225°)
ne_sector = in_sector(launch, features, 0, 90)
s_sector  = in_sector(launch, features, 135, 225)

print("Northeast sector (0° – 90°):")
for feat, brg in ne_sector:
    print(f"  {feat['properties']['name']:<26} {brg:.1f}°")

print()
print("South sector (135° – 225°):")
for feat, brg in s_sector:
    print(f"  {feat['properties']['name']:<26} {brg:.1f}°")

Northeast sector (0° – 90°):
  Oklahoma City              26.8°
  Tinker AFB                 31.6°
  Fort Smith Regional        66.0°

South sector (135° – 225°):
  Kelly Field Annex          179.8°


## 3. Drawing Direction Arrows on a Map

A bearing is more intuitive as an arrow than a number. The function below builds a short GeoJSON LineString that points in the bearing direction from a given origin — suitable for adding to an ipyleaflet map as a direction indicator.

In [5]:
def destination_point(origin, bearing_deg, distance_km):
    """
    Compute the point reached by travelling distance_km from origin at bearing_deg.
    Uses the spherical direct formula.
    Inputs: origin [lon, lat], bearing in degrees, distance in km.
    Returns: [lon, lat]
    """
    R = 6371.0
    d = distance_km / R  # angular distance in radians
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])

    lat2 = math.asin(
        math.sin(lat1) * math.cos(d) +
        math.cos(lat1) * math.sin(d) * math.cos(brg)
    )
    lon2 = lon1 + math.atan2(
        math.sin(brg) * math.sin(d) * math.cos(lat1),
        math.cos(d) - math.sin(lat1) * math.sin(lat2)
    )
    return [math.degrees(lon2), math.degrees(lat2)]


def bearing_arrow(origin, bearing_deg, length_km=50):
    """
    Build a GeoJSON LineString Feature pointing from origin in the given bearing direction.
    """
    tip = destination_point(origin, bearing_deg, length_km)
    return {
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [origin, tip]},
        "properties": {"bearing": round(bearing_deg, 1)}
    }


# Draw a bearing arrow to every target from Wichita Falls
arrow_features = []
for feat in features:
    coords = feat["geometry"]["coordinates"]
    brg = compute_bearing(launch, coords)
    arrow_features.append(bearing_arrow(launch, brg, length_km=60))

arrows_fc  = {"type": "FeatureCollection", "features": arrow_features}
targets_fc_display = targets_fc

m2 = Map(center=(33.91, -98.49), zoom=6)
m2.add(GeoJSON(data=targets_fc_display,
               point_style={"radius": 5, "color": "#457b9d", "fillOpacity": 0.9}))
m2.add(GeoJSON(data=arrows_fc,
               style={"color": "#e63946", "weight": 2}))
m2

Map(center=[33.91, -98.49], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

Notice that `destination_point` is the inverse of `compute_bearing`: given an origin, a direction, and a distance, it returns the endpoint. This is the **direct problem** in geodesy — and it is how you will plot range rings and projected positions in the next notebook.

## 4. Simple Targeting Picture

Bearing and distance together define a targeting picture: for a given launch site, which targets are reachable, and in what direction does each one lie?

The map below combines everything: the launch site, all targets, bearing arrows, and a range ring at a given radius. Targets inside the ring are highlighted; targets outside are dimmed.

In [6]:
def range_ring(center, radius_km, n=64):
    """Build a GeoJSON Polygon approximating a circle of radius_km around center."""
    coords = [destination_point(center, 360 * i / n, radius_km) for i in range(n)]
    coords.append(coords[0])  # close the ring
    return {
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [coords]},
        "properties": {"radius_km": radius_km}
    }


MAX_RANGE_KM = 400

in_range_features  = []
out_range_features = []
line_features      = []

for feat in features:
    coords = feat["geometry"]["coordinates"]
    dist = haversine_km(launch, coords)
    brg  = compute_bearing(launch, coords)

    if dist <= MAX_RANGE_KM:
        in_range_features.append(feat)
        # Draw a line from launch to target
        line_features.append({
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": [launch, coords]},
            "properties": {"bearing": round(brg, 1), "distance_km": round(dist, 1)}
        })
    else:
        out_range_features.append(feat)

ring_fc     = {"type": "FeatureCollection", "features": [range_ring(launch, MAX_RANGE_KM)]}
in_fc       = {"type": "FeatureCollection", "features": in_range_features}
out_fc      = {"type": "FeatureCollection", "features": out_range_features}
lines_fc    = {"type": "FeatureCollection", "features": line_features}
launch_fc   = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": launch},
     "properties": {"name": "Wichita Falls (launch)"}}
]}

m3 = Map(center=(33.91, -98.49), zoom=6)
m3.add(GeoJSON(data=ring_fc,   style={"color": "#e63946", "weight": 1.5, "fillOpacity": 0.05}))
m3.add(GeoJSON(data=lines_fc,  style={"color": "#e63946", "weight": 1.5, "dashArray": "4"}))
m3.add(GeoJSON(data=in_fc,     point_style={"radius": 6, "color": "#2a9d8f", "fillOpacity": 1.0}))
m3.add(GeoJSON(data=out_fc,    point_style={"radius": 6, "color": "#aaaaaa", "fillOpacity": 0.5}))
m3.add(GeoJSON(data=launch_fc, point_style={"radius": 7, "color": "#e63946", "fillOpacity": 1.0}))

print(f"Launch site: Wichita Falls   Max range: {MAX_RANGE_KM} km")
print(f"Targets in range:  {len(in_range_features)}")
print(f"Targets out of range: {len(out_range_features)}")
print()
for feat in in_range_features:
    coords = feat["geometry"]["coordinates"]
    print(f"  {feat['properties']['name']:<26}"
          f"  {compute_bearing(launch, coords):.1f}°  "
          f"  {haversine_km(launch, coords):.0f} km")
m3

Launch site: Wichita Falls   Max range: 400 km
Targets in range:  5
Targets out of range: 2

  Tinker AFB                  31.6°    194 km
  Lubbock Preston Smith       265.5°    298 km
  NAS Fort Worth JRB          130.8°    179 km
  Altus AFB                   319.5°    110 km
  Oklahoma City               26.8°    195 km


Map(center=[33.91, -98.49], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

Green targets are in range. Gray targets are not. The dashed lines from the launch site show the exact bearing to each reachable target. Adjust `MAX_RANGE_KM` and rerun to see the picture change.

---

## Exercise A — Interactive Range Slider

Wire the targeting picture to an `ipywidgets.IntSlider` so that changing the range updates which targets are highlighted and which lines are drawn, without rewriting the map from scratch.

In [10]:
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import Map, GeoJSON

slider = widgets.IntSlider(
    value=300,
    min=50,
    max=800,
    step=50,
    description="Range (km):"
)

out = widgets.Output()

m = Map(center=(33.91, -98.49), zoom=6)

dynamic_layers = []

def make_target_features(max_range):

    in_range_features  = []
    out_range_features = []
    line_features      = []

    # FIX: use features, not targets
    for feat in features:

        coords = feat["geometry"]["coordinates"]

        dist = haversine_km(launch, coords)
        brg  = compute_bearing(launch, coords)

        if dist <= max_range:

            in_range_features.append(feat)

            line_features.append({
                "type": "Feature",
                "geometry": {
                    "type": "LineString",
                    "coordinates": [launch, coords]
                },
                "properties": {
                    "bearing": round(brg, 1),
                    "distance_km": round(dist, 1)
                }
            })

        else:
            out_range_features.append(feat)

    ring_fc = {
        "type": "FeatureCollection",
        "features": [range_ring(launch, max_range)]
    }

    in_fc = {
        "type": "FeatureCollection",
        "features": in_range_features
    }

    out_fc = {
        "type": "FeatureCollection",
        "features": out_range_features
    }

    lines_fc = {
        "type": "FeatureCollection",
        "features": line_features
    }

    return ring_fc, in_fc, out_fc, lines_fc


def update_range(change):

    global dynamic_layers

    max_range = change["new"]

    # remove old layers
    for layer in dynamic_layers:
        m.remove(layer)

    dynamic_layers.clear()

    ring_fc, in_fc, out_fc, lines_fc = make_target_features(max_range)

    ring_layer = GeoJSON(
        data=ring_fc,
        style={
            "color": "#e63946",
            "weight": 1.5,
            "fillOpacity": 0.05
        }
    )

    line_layer = GeoJSON(
        data=lines_fc,
        style={
            "color": "#e63946",
            "weight": 1.5,
            "dashArray": "4"
        }
    )

    in_layer = GeoJSON(
        data=in_fc,
        point_style={
            "radius": 6,
            "color": "#2a9d8f",
            "fillOpacity": 1.0
        }
    )

    out_layer = GeoJSON(
        data=out_fc,
        point_style={
            "radius": 6,
            "color": "#aaaaaa",
            "fillOpacity": 0.5
        }
    )

    m.add(ring_layer)
    m.add(line_layer)
    m.add(in_layer)
    m.add(out_layer)

    dynamic_layers.extend([
        ring_layer,
        line_layer,
        in_layer,
        out_layer
    ])

    with out:

        out.clear_output()

        print(f"Launch site: Wichita Falls")
        print(f"Max range: {max_range} km\n")

        print(f"Targets in range: {len(in_fc['features'])}")
        print(f"Targets out of range: {len(out_fc['features'])}\n")

        for feat in in_fc["features"]:

            coords = feat["geometry"]["coordinates"]

            print(
                f"  {feat['properties']['name']:<26}"
                f" {compute_bearing(launch, coords):.1f}°"
                f"   {haversine_km(launch, coords):.0f} km"
            )


slider.observe(update_range, names="value")

update_range({"new": slider.value})

display(slider, out, m)

IntSlider(value=300, description='Range (km):', max=800, min=50, step=50)

Output()

Map(center=[33.91, -98.49], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Exercise B — Sector + Range Combined

Write a function `targets_in_envelope(origin, features, start_bearing, end_bearing, max_range_km)` that returns only targets that satisfy **both** constraints: within the bearing sector **and** within the range.

Test it with a northeast sector (0°–90°) at 500 km from Wichita Falls. Print the name, bearing, and distance of each result.

In [11]:
def targets_in_envelope(origin, features, start_bearing, end_bearing, max_range_km):
    results = []

    for feat in features:
        coords = feat["geometry"]["coordinates"]

        brg = compute_bearing(origin, coords)
        dist = haversine_km(origin, coords)

        # Handle normal sector and wraparound sector
        if start_bearing <= end_bearing:
            in_sector = start_bearing <= brg <= end_bearing
        else:
            in_sector = brg >= start_bearing or brg <= end_bearing

        in_range = dist <= max_range_km

        if in_sector and in_range:
            results.append((feat, brg, dist))

    return sorted(results, key=lambda x: x[2])


results = targets_in_envelope(launch, features, 0, 90, 500)

print("Targets in NE sector within 500 km:\n")

for feat, brg, dist in results:
    print(f"  {feat['properties']['name']:<26}  {brg:.1f}°  {dist:.0f} km")

Targets in NE sector within 500 km:

  Tinker AFB                  31.6°  194 km
  Oklahoma City               26.8°  195 km
  Fort Smith Regional         66.0°  409 km


## Exercise C — Nearest Target by Direction

Given a desired heading of `45°` (northeast), find the target whose actual bearing from the launch site is **closest to that heading**, regardless of distance.

Write a function `nearest_by_bearing(origin, features, desired_bearing)` that returns the single closest-bearing target. Account for wraparound so that a target at `359°` is considered close to a desired bearing of `1°`.

In [12]:
def bearing_diff(a, b):
    """Shortest angular difference between two bearings, in [0, 180]."""
    diff = abs(a - b) % 360
    return min(diff, 360 - diff)


def nearest_by_bearing(origin, features, desired_bearing):
    best = None
    best_diff = float("inf")

    for feat in features:
        coords = feat["geometry"]["coordinates"]

        brg = compute_bearing(origin, coords)
        diff = bearing_diff(brg, desired_bearing)

        if diff < best_diff:
            best_diff = diff
            best = (feat, brg, diff)

    return best


result = nearest_by_bearing(launch, features, 45)

if result:
    feat, brg, diff = result
    print(
        f"Nearest to 45°: {feat['properties']['name']} "
        f"({brg:.1f}°, difference {diff:.1f}°)"
    )

Nearest to 45°: Tinker AFB (31.6°, difference 13.4°)


---

## Check Your Understanding

A student writes this sector filter:

```python
def in_sector_buggy(origin, features, start_bearing, end_bearing):
    results = []
    for feat in features:
        coords = feat["geometry"]["coordinates"]
        brg = compute_bearing(origin, coords)
        if start_bearing <= brg <= end_bearing:
            results.append((feat, brg))
    return results
```

They test it with a sector from `10°` to `80°` (a narrow NE wedge) and it works correctly. Then a colleague uses it with a sector from `350°` to `30°` (a narrow North wedge spanning 0°) and gets zero results — even though two targets clearly lie due north.

**Question:** What is the bug, and what is the minimal fix? What property of compass bearings does the original code fail to account for?

```python
# your answer here
```


---

## Check Your Understanding

The bug is that the original sector filter only works when the starting bearing is less than the ending bearing, such as `10°` to `80°`.

That works for a normal sector, but it fails for a sector like `350°` to `30°` because that sector crosses `0°`.

A bearing of `355°` should count, and a bearing of `10°` should also count, but the condition:

```python
start_bearing <= brg <= end_bearing
```

becomes:

```python
350 <= brg <= 30
```

which can never be true.

The minimal fix is to check whether the sector wraps around `0°`:

```python
if start_bearing <= end_bearing:
    in_sector = start_bearing <= brg <= end_bearing
else:
    in_sector = brg >= start_bearing or brg <= end_bearing
```

The original code fails to account for the circular nature of compass bearings. On a compass, `359°` is right next to `0°`, not far away from it.

## Next

In [04 — Advanced Bearing](./04-Advanced_Bearing.ipynb), we cover destination point computation, midpoint bearing, and the difference between initial and final bearing on a great-circle path.